In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


Client ready: True


In [2]:
from embedder import Embedder

embedder = Embedder()
v = embedder.encode("test query")
print("Embedding shape:", v.shape)

Embedding shape: (384,)


In [3]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Documents: {len(documents)}")   
print(f"Chunks: {len(chunks)}")

Documents: 72
Chunks: 295


In [4]:
from pydantic import BaseModel
from evaluation_utils import llm_structured
import json

class Questions(BaseModel):
    questions: list[str]

instructions = (
    "You emulate a student who is taking our LLM course. You are given one "
    "lesson page from the course. Formulate 5 questions this student might ask "
    "that are answered by this page. Rules: The page should contain the answer "
    "to each question. Make the questions complete and not too short. Use as "
    "few words as possible from the page; don't copy its phrasing. The questions "
    "should resemble how people actually ask things online: not too formal, not "
    "too short, not too long. Ask about the content of the lesson, not about its "
    "formatting or filename."
)

In [5]:
print(documents[0].keys() if hasattr(documents[0], "keys") else type(documents[0]))

sample_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
sample_pages = [d for d in documents if d["filename"] in sample_filenames]
print("Matched pages:", len(sample_pages))

dict_keys(['content', 'filename'])
Matched pages: 3


In [6]:
input_tokens = []
ground_truth_q1 = []

for page in sample_pages:
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"],
    })
    parsed, usage = llm_structured(client, instructions, user_prompt, Questions)

    input_tokens.append(usage.input_tokens)
    for q in parsed.questions:
        ground_truth_q1.append({"question": q, "filename": page["filename"]})

avg_input_tokens = sum(input_tokens) / 3
print("Per-call input tokens:", input_tokens)
print("Average input tokens:", avg_input_tokens)

Per-call input tokens: [1015, 1281, 1748]
Average input tokens: 1348.0


In [7]:
import pandas as pd

In [8]:
df_gt = pd.read_csv("ground-truth.csv")
ground_truth = df_gt.to_dict(orient="records")
print("Ground truth questions:", len(ground_truth))   
print("First question:", ground_truth[0]["question"])

Ground truth questions: 360
First question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [9]:
import minsearch
from minsearch import VectorSearch

# text index
tindex = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

# vector index
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

def text_search(query, num_results=5):
    return tindex.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    q_vec = embedder.encode(query)
    return vindex.search(q_vec, num_results=num_results)
print("Indexes ready.")

Indexes ready.


In [10]:
q = ground_truth[0]["question"]

text_results = text_search(q)
print("Q2 top filename:", text_results[0]["filename"])

Q2 top filename: 01-agentic-rag/lessons/03-rag.md


In [11]:
vector_results = vector_search(q)
print("Q3 top filename:", vector_results[0]["filename"])

Q3 top filename: 01-agentic-rag/lessons/01-intro.md


In [12]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt += 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in ground_truth:
        target = q["filename"]
        results = search_function(q["question"])
        relevance = [d["filename"] == target for d in results]
        relevance_total.append(relevance)
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}


In [14]:
q4 = evaluate(ground_truth, text_search)
print("Q4 text search:", q4)

Q4 text search: {'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


In [15]:
q5 = evaluate(ground_truth, vector_search)
print("Q5 vector search:", q5)

Q5 vector search: {'hit_rate': 0.725, 'mrr': 0.5486111111111112}


In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [17]:
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda q, k=k: hybrid_search(q, k=k))
    print(f"k={k:<4} -> MRR={result['mrr']:.4f}  HitRate={result['hit_rate']:.4f}")

k=1    -> MRR=0.6482  HitRate=0.8389
k=50   -> MRR=0.6379  HitRate=0.8361
k=100  -> MRR=0.6379  HitRate=0.8361
k=200  -> MRR=0.6379  HitRate=0.8361
